# Aula 1 - LLM APIs

## Template da V1 do projeto

Neste notebook, você vai montar a primeira versão do assistente de estudos da monitoria.

As células com `TODO` são partes que você precisa completar. Não coloque sua API key diretamente no notebook, utilize o `.env`.

## Objetivos

Ao final, você deve conseguir:

- configurar um cliente para a API da Groq;
- montar mensagens com `system` e `user`;
- enviar uma requisição e ler a resposta;
- observar o uso de tokens;
- explicar por que chamadas separadas não têm memória automática;
- testar o efeito da temperatura;
- salvar a V1 em um commit.

## 1. Configuração do ambiente

Antes de executar o notebook:

1. crie um ambiente virtual;
2. instale as dependências de `requirements.txt`;
3. copie `env.example` para `.env`;
4. preencha `GROQ_API_KEY` no `.env`.

In [2]:
# Passo a passo para criar o .venv: 
# Abra o Terminal na raiz do projeto e digite:
# python -m venv venv
# Depois digite: 
# cd "Aula 01"
# pip install -r requirements.txt

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL","https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Monte as mensagens

A aplicação controla o comportamento do assistente usando mensagens.

Use uma mensagem `system` para definir o papel do assistente e uma mensagem `user` para fazer uma pergunta sobre a monitoria.

In [3]:
# TODO: crie uma lista chamada `messages` com uma mensagem `system` e uma mensagem `user`.
messages = [
    {
        "role": "system",
        "content":"Você é um assistente de estudos. Explique com clareza e use exemplos curtos.",
    },
    {
        "role":"user",
        "content":"Explique o que é uma LLM em duas frases."
    }
]


## 3. Faça a primeira chamada

Envie `messages` para o modelo configurado em `MODEL` e salve a resposta em uma variável chamada `response`.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
)

## 4. Leia o texto da resposta

A resposta contém uma lista de escolhas. Nesta atividade, vamos usar o texto da primeira escolha.

In [5]:

print(response.choices[0].message.content)

Uma LLM (Large Language Model) é um modelo de inteligência artificial treinado com enormes volumes de texto para prever e gerar linguagem natural de forma coerente. Por exemplo, ao receber a frase “O céu está”, a LLM pode completar “azul e cheio de nuvens” usando o conhecimento adquirido durante o treinamento.


## 5. Observe o uso de tokens

Além do texto, a API informa quantos tokens entraram na chamada, quantos saíram e qual foi o total.

In [13]:
# TODO: mostre `prompt_tokens`, `completion_tokens` e `total_tokens` usando `response.usage`.
usage = response.usage

print(f"Prompt tokens:     {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens:      {usage.total_tokens}")

Prompt tokens:     105
Completion tokens: 141
Total tokens:      246


## 6. Teste a ausência de memória

Faça uma chamada dizendo que o nome do seu usuário. Depois faça uma segunda chamada perguntando o nome, mas sem reenviar a primeira mensagem.

Salve a segunda resposta em `sem_historico`. O que aconteceu?

In [7]:
# TODO: faça duas chamadas independentes e salve a segunda resposta em `sem_historico`.
client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Olá, meu nome é Rodrigo."}],
    temperature=0,
)

# 2ª chamada: pergunta o nome sem enviar o histórico anterior
sem_historico = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Qual é o meu nome?"}],
    temperature=0,
)

In [8]:
print(sem_historico.choices[0].message.content)

Não sei qual é o seu nome, pois você ainda não o compartilhou comigo. Se quiser me dizer, ficarei feliz em usá‑lo nas próximas conversas!


Agora repita a pergunta, mas envie explicitamente o histórico em `messages`. Inclua uma mensagem `assistant` representando a resposta anterior.

A aplicação cria a sensação de memória ao reenviar o contexto.

In [9]:
# TODO: crie uma nova lista de mensagens com a conversa completa e salve a resposta em `com_historico`.
mensagens_com_historico = [
    {"role": "user", "content": "Olá, meu nome é Rodrigo."},
    {"role": "assistant", "content": "Olá, Rodrigo! Como posso te ajudar hoje?"},
    {"role": "user", "content": "Qual é o meu nome?"},
]

com_historico = client.chat.completions.create(
    model=MODEL,
    messages=mensagens_com_historico,
    temperature=0,
)

In [10]:
# TODO: mostre o texto de `com_historico` e compare com o resultado sem histórico.
print(com_historico.choices[0].message.content)

Seu nome é Rodrigo.


## 7. Experimente a temperatura

Use o mesmo prompt duas vezes: uma com `temperature=0` e outra com `temperature=1`.

Compare a variedade das respostas. Temperatura baixa tende a ser mais previsível; temperatura alta tende a permitir mais variação.

In [11]:
# TODO: faça as chamadas com duas temperaturas e compare os resultados.
prompt_temperatura = [
    {
        "role": "user",
        "content": "Crie 3 ideias criativas de nomes para um projeto de assistente de estudos com IA."
    }
]

# Chamada com temperature = 0 (determinística e focada)
resp_temp_0 = client.chat.completions.create(
    model=MODEL,
    messages=prompt_temperatura,
    temperature=0,
)

# Chamada com temperature = 1 (mais diversa e criativa)
resp_temp_1 = client.chat.completions.create(
    model=MODEL,
    messages=prompt_temperatura,
    temperature=1,
)

print("=== Resposta com Temperature = 0 ===")
print(resp_temp_0.choices[0].message.content)
print("\n" + "="*40 + "\n")
print("=== Resposta com Temperature = 1 ===")
print(resp_temp_1.choices[0].message.content)

=== Resposta com Temperature = 0 ===
Claro! Aqui vão três sugestões de nomes criativos para o seu projeto de assistente de estudos com IA:

| # | Nome | Por que funciona |
|---|------|-------------------|
| 1 | **MentorMind** | Une a ideia de “mentor” (guia experiente) com “mind” (mente), transmitindo a sensação de um parceiro inteligente que acompanha o processo de aprendizagem. |
| 2 | **EstudaBot** | Simples e direto, combina “estuda” (atividade central) com “bot”, deixando claro que se trata de um assistente automatizado e tecnológico. |
| 3 | **SapiensAI** | Faz referência ao ser humano pensante (“Homo sapiens”) e à inteligência artificial, sugerindo um aliado que eleva o potencial cognitivo do usuário. |

Qualquer uma dessas opções pode ser adaptada com um slogan ou um logotipo que reflita a personalidade do seu assistente. Boa sorte com o projeto!


=== Resposta com Temperature = 1 ===
Claro! Aqui vão três sugestões de nomes criativos para o seu projeto de assistente de estudos 

## 8. Checkpoint da V1

Antes de fazer o commit, confirme:

- [ ] a aplicação chamou a API da Groq;
- [ ] a chave ficou no `.env`;
- [ ] a requisição usou `system` e `user`;
- [ ] a resposta apareceu;
- [ ] o uso de tokens foi observado;
- [ ] você comparou chamadas sem histórico e com histórico;
- [ ] você testou duas temperaturas.
